# Análise de Vendas e Comportamento dos Clientes — Olist

### Projeto de Análise Exploratória de Dados

**Ferramentas:** Python • Pandas • Matplotlib

---

## Objetivo

Investigar o desempenho comercial do marketplace Olist a partir de três perspectivas principais:

- **Vendas:** volume de pedidos, receita e evolução ao longo do tempo;
- **Produtos e categorias:** itens mais vendidos, maior receita e ticket médio;
- **Clientes e localização:** concentração geográfica, cidades e relação entre pedidos e receita.

> **Pergunta central:** quais padrões de vendas, produtos, categorias e mercados geográficos se destacam nos dados da Olist?

## 1. Dados utilizados

O projeto utiliza as principais tabelas do dataset público da Olist:

- `olist_orders_dataset.csv` — informações dos pedidos;
- `olist_order_items_dataset.csv` — itens, preços e frete;
- `olist_products_dataset.csv` — informações dos produtos e categorias;
- `olist_customers_dataset.csv` — localização dos clientes.

### Relacionamentos principais

- `order_id` → relaciona pedidos e itens;
- `customer_id` → relaciona pedidos e clientes;
- `product_id` → relaciona itens e produtos.

A análise será construída progressivamente, integrando essas informações para gerar indicadores de negócio.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

orders = pd.read_csv('/content/olist_orders_dataset.csv')
order_items = pd.read_csv('/content/olist_order_items_dataset.csv')
products = pd.read_csv('/content/olist_products_dataset.csv')
customers = pd.read_csv('/content/olist_customers_dataset.csv')

print('Pedidos:', orders.shape)
print('Itens dos pedidos:', order_items.shape)
print('Produtos:', products.shape)
print('Clientes:', customers.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/content/olist_orders_dataset.csv'

## 2.  Qualidade e estrutura dos dados

Antes de calcular os indicadores, verificamos dimensões, tipos de dados e valores ausentes.

Essa etapa é importante porque problemas de qualidade podem afetar os resultados das análises posteriores.

In [ ]:
print('--- ORDERS ---')
orders.info()

print('\n--- ORDER ITEMS ---')
order_items.info()

print('\n--- PRODUCTS ---')
products.info()

print('\n--- CUSTOMERS ---')
customers.info()

In [ ]:
print('Valores ausentes por tabela:')
print('\nOrders:')
display(orders.isnull().sum().sort_values(ascending=False).head(10))

print('Products:')
display(products.isnull().sum().sort_values(ascending=False).head(10))

print('Customers:')
display(customers.isnull().sum().sort_values(ascending=False).head(10))

### Observação sobre categorias

Na base de produtos existem registros sem categoria. Na análise de receita por categoria, esses registros serão excluídos apenas dessa agregação, preservando os demais dados de vendas.

Essa decisão evita atribuir uma categoria artificial aos produtos sem classificação.

## 3.  Indicadores gerais de receita

Primeiro, calculamos o valor total dos produtos, o valor total do frete e o valor movimentado considerando ambos.

In [ ]:
valor_produtos = order_items['price'].sum()
valor_frete = order_items['freight_value'].sum()
valor_total = valor_produtos + valor_frete

print(f'Valor total dos produtos: R$ {valor_produtos:,.2f}')
print(f'Valor total do frete: R$ {valor_frete:,.2f}')
print(f'Valor total considerando produtos + frete: R$ {valor_total:,.2f}')

###  Insight

Os dados analisados apresentam aproximadamente **R$ 13,59 milhões em produtos** e **R$ 2,25 milhões em frete**, totalizando cerca de **R$ 15,84 milhões** quando os dois componentes são considerados.

Isso mostra que o frete representa uma parcela relevante do valor movimentado e merece atenção em análises comerciais e logísticas.

## 4.  Evolução mensal dos pedidos

A dimensão temporal permite identificar períodos de maior e menor movimentação e observar a evolução do volume de pedidos.

In [ ]:
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp']
)

pedidos_mes = (
    orders.assign(
        ano_mes=orders['order_purchase_timestamp'].dt.to_period('M')
    )
    .groupby('ano_mes')
    .size()
)

plt.figure(figsize=(12, 6))
plt.plot(pedidos_mes.index.astype(str), pedidos_mes.values, marker='o')
plt.title('Evolução mensal dos pedidos — Olist')
plt.xlabel('Mês')
plt.ylabel('Quantidade de pedidos')
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

###  Como interpretar

O gráfico permite observar a trajetória do volume de pedidos ao longo do período disponível na base.

Para uma análise de negócio, os próximos passos podem envolver a identificação de picos sazonais, períodos de crescimento e possíveis quedas de demanda.

## 5.  Desempenho dos produtos

Nesta etapa analisamos duas perguntas diferentes:

1. **Quais produtos geraram mais receita?**
2. **Quais produtos foram vendidos em maior quantidade?**

Essas métricas não são equivalentes: um produto pode vender poucas unidades e gerar muita receita, enquanto outro pode ter grande volume de unidades e receita menor.

In [ ]:
vendas_produto = (
    order_items
    .groupby('product_id')
    .agg(
        quantidade=('product_id', 'count'),
        receita=('price', 'sum')
    )
    .sort_values('receita', ascending=False)
)

print('Top 10 produtos por receita:')
display(vendas_produto.head(10))

print('Top 10 produtos por quantidade:')
display(
    vendas_produto
    .sort_values('quantidade', ascending=False)
    .head(10)
)

###  Insight

A comparação entre os dois rankings evidencia que **volume de vendas e receita são dimensões diferentes de desempenho**.

Por isso, uma avaliação comercial não deve considerar somente a quantidade vendida: é importante observar também o valor financeiro gerado.

## 6.  Receita por categoria

Agora combinamos os resultados de vendas com as informações de produtos para identificar quais categorias apresentam maior receita.

In [ ]:
vendas_detalhadas = (
    vendas_produto
    .reset_index()
    .merge(
        products[['product_id', 'product_category_name']],
        on='product_id',
        how='left'
    )
)

print(
    'Registros de produtos sem categoria:',
    vendas_detalhadas['product_category_name'].isnull().sum()
)

receita_categoria = (
    vendas_detalhadas
    .dropna(subset=['product_category_name'])
    .groupby('product_category_name')
    .agg(
        quantidade=('quantidade', 'sum'),
        receita=('receita', 'sum')
    )
    .sort_values('receita', ascending=False)
)

display(receita_categoria.head(10))

In [ ]:
top_categorias = receita_categoria.head(10)

plt.figure(figsize=(12, 7))
plt.barh(
    top_categorias.index[::-1],
    top_categorias['receita'].values[::-1]
)
plt.title('Top 10 categorias por receita — Olist')
plt.xlabel('Receita (R$)')
plt.ylabel('Categoria')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

###  Insight

Entre as categorias apresentadas nos resultados, **beleza_saude** lidera em receita, seguida por **relogios_presentes** e **cama_mesa_banho**.

Essas categorias se destacam como importantes fontes de faturamento e podem ser priorizadas em análises posteriores de produtos, clientes e sazonalidade.

## 7.  Ticket médio por categoria

O ticket médio é calculado como:

**Ticket médio = receita ÷ quantidade vendida**

Para evitar que categorias com poucas observações distorçam a comparação, consideramos apenas categorias com pelo menos **500 unidades vendidas**.

In [ ]:
ticket_categoria = receita_categoria.copy()
ticket_categoria['ticket_medio'] = (
    ticket_categoria['receita'] / ticket_categoria['quantidade']
)

ticket_categoria_relevante = (
    ticket_categoria[ticket_categoria['quantidade'] >= 500]
    .sort_values('ticket_medio', ascending=False)
)

display(ticket_categoria_relevante.head(10))

###  Insight

Entre as categorias com volume relevante, **instrumentos_musicais** e **eletroportateis** apresentam os maiores tickets médios nos resultados obtidos.

Isso indica que categorias com menor receita total podem, ainda assim, apresentar produtos de maior valor médio por venda.

## 8.  Distribuição dos clientes por estado

Agora deslocamos a análise para o perfil geográfico dos clientes.

In [ ]:
clientes_estado = (
    customers
    .groupby('customer_state')
    .size()
    .reset_index(name='clientes')
    .sort_values('clientes', ascending=False)
)

clientes_estado['percentual'] = (
    clientes_estado['clientes'] /
    clientes_estado['clientes'].sum() * 100
)

display(clientes_estado.head(10))

top3_clientes = (
    clientes_estado.head(3)['clientes'].sum() /
    clientes_estado['clientes'].sum() * 100
)

print(
    f'Os 3 principais estados representam '
    f'{top3_clientes:.2f}% dos clientes.'
)

###  Insight

**São Paulo, Rio de Janeiro e Minas Gerais** concentram **66,61% dos clientes**.

São Paulo sozinho possui a maior concentração da base, indicando forte dependência dos principais mercados consumidores do Sudeste.

## 9.  Receita por estado

Para relacionar localização e desempenho financeiro, integramos pedidos, clientes e itens dos pedidos.

In [ ]:
pedidos_clientes = orders.merge(
    customers[['customer_id', 'customer_state', 'customer_city']],
    on='customer_id',
    how='left'
)

vendas_estado = pedidos_clientes.merge(
    order_items[['order_id', 'price', 'freight_value']],
    on='order_id',
    how='inner'
)

receita_estado = (
    vendas_estado
    .groupby('customer_state')
    .agg(
        vendas=('order_id', 'nunique'),
        quantidade=('order_id', 'count'),
        receita=('price', 'sum'),
        frete=('freight_value', 'sum')
    )
    .sort_values('receita', ascending=False)
)

receita_estado['ticket_medio'] = (
    receita_estado['receita'] / receita_estado['vendas']
)

display(receita_estado.head(10))

top3_receita = (
    receita_estado.head(3)['receita'].sum() /
    receita_estado['receita'].sum() * 100
)

sp_receita = (
    receita_estado.loc['SP', 'receita'] /
    receita_estado['receita'].sum() * 100
)

print(f'Os 3 principais estados representam {top3_receita:.2f}% da receita.')
print(f'São Paulo representa {sp_receita:.2f}% da receita total.')

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

top_estados = clientes_estado.head(10).copy()

ax1.bar(
    top_estados['customer_state'],
    top_estados['clientes']
)
ax1.set_xlabel('Estado')
ax1.set_ylabel('Número de clientes')

ax2 = ax1.twinx()

receita_top = receita_estado.loc[
    top_estados['customer_state'], 'receita'
]

ax2.plot(
    top_estados['customer_state'],
    receita_top.values,
    marker='o'
)
ax2.set_ylabel('Receita (R$)')

plt.title('Concentração de clientes e receita por estado')
plt.tight_layout()
plt.show()

###  Insight

A análise mostra uma forte concentração tanto de clientes quanto de receita nos principais estados.

**São Paulo** aparece como o principal mercado, representando aproximadamente **38,28% da receita total** nos resultados calculados.

## 10.  Análise por cidade

Estados mostram a concentração macrogeográfica. A análise por cidade permite identificar os principais mercados urbanos.

In [ ]:
clientes_cidade = (
    customers
    .groupby('customer_city')
    .size()
    .reset_index(name='clientes')
    .sort_values('clientes', ascending=False)
)

display(clientes_cidade.head(15))

receita_cidade = (
    vendas_estado
    .groupby('customer_city')
    .agg(
        pedidos=('order_id', 'nunique'),
        receita=('price', 'sum')
    )
)

receita_cidade['ticket_medio'] = (
    receita_cidade['receita'] /
    receita_cidade['pedidos']
)

receita_cidade = receita_cidade.sort_values(
    'receita',
    ascending=False
)

display(receita_cidade.head(15))

 Insight

**São Paulo** lidera também na análise por cidade, com aproximadamente **15,5 mil clientes**, mais de **15,4 mil pedidos** e cerca de **R$ 1,91 milhão em receita** nos resultados obtidos.

Rio de Janeiro e Belo Horizonte aparecem em seguida, reforçando a importância dos grandes centros urbanos.

## 11. Relação entre volume de pedidos e receita

O gráfico de dispersão ajuda a identificar se cidades com mais pedidos também tendem a gerar mais receita.

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    receita_cidade['pedidos'],
    receita_cidade['receita']
)

plt.xlabel('Quantidade de pedidos')
plt.ylabel('Receita (R$)')
plt.title('Relação entre volume de pedidos e receita por cidade')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

###  Interpretação

Existe uma relação positiva evidente: cidades com maior volume de pedidos tendem a apresentar maior receita.

Entretanto, os pontos não se distribuem exatamente sobre uma mesma linha, mostrando que **ticket médio também influencia o resultado financeiro**.

## 12.  Cidades com maior ticket médio

Cidades com poucos pedidos podem apresentar tickets médios artificialmente elevados. Por isso, aplicamos um filtro mínimo de **100 pedidos**.

In [ ]:
ticket_cidade_relevante = (
    receita_cidade[receita_cidade['pedidos'] >= 100]
    .sort_values('ticket_medio', ascending=False)
)

display(ticket_cidade_relevante.head(15))

In [ ]:
top_ticket_cidade = ticket_cidade_relevante.head(10)

plt.figure(figsize=(10, 6))
plt.barh(
    top_ticket_cidade.index[::-1],
    top_ticket_cidade['ticket_medio'].values[::-1]
)

plt.xlabel('Ticket médio (R$)')
plt.ylabel('Cidade')
plt.title('Cidades com maior ticket médio — mínimo de 100 pedidos')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

###  Insight

Com o filtro de volume mínimo, **Divinópolis, João Pessoa e Porto Velho** aparecem entre as cidades de maior ticket médio nos resultados apresentados.

O filtro de 100 pedidos torna essa comparação mais robusta do que simplesmente ordenar todas as cidades, pois reduz o efeito de cidades com apenas uma ou poucas vendas.

# 13.  Principais insights do projeto

### 1. Forte concentração geográfica
São Paulo, Rio de Janeiro e Minas Gerais concentram **66,61% dos clientes**.

### 2. São Paulo é o principal mercado
O estado representa aproximadamente **38,28% da receita total**.

### 3. Receita não depende apenas de volume
Produtos e categorias podem apresentar posições diferentes quando avaliados por quantidade vendida e receita.

### 4. Beleza e saúde lidera em receita
A categoria **beleza_saude** apresentou a maior receita entre as categorias analisadas.

### 5. Algumas categorias possuem ticket médio elevado
Entre categorias com pelo menos 500 unidades vendidas, **instrumentos_musicais** e **eletroportateis** se destacam.

### 6. Grandes centros concentram o faturamento
São Paulo, Rio de Janeiro e Belo Horizonte aparecem entre os principais mercados urbanos.

### 7. Ticket médio complementa a análise de volume
A análise por cidade mostra que mercados menores podem apresentar ticket médio elevado, mesmo sem possuir o maior número de pedidos.

# 14.  Conclusão

A análise exploratória evidencia três características importantes do marketplace:

**concentração geográfica + diversidade de categorias + diferença entre volume e valor.**

Os resultados indicam que os principais mercados consumidores estão concentrados no Sudeste, especialmente em São Paulo. Ao mesmo tempo, a análise de produtos e categorias mostra que desempenho comercial não deve ser avaliado apenas pela quantidade vendida.

Para uma análise de negócio mais completa, os próximos estudos podem explorar:

- avaliação dos clientes;
- prazo e atraso das entregas;
- custo e participação do frete;
- vendedores;
- formas de pagamento;
- comportamento temporal das categorias;
- segmentação de clientes;
- construção de dashboard interativo.

---

##  Autor

**Elias Trindade**

Projeto de portfólio em **Análise de Dados / Business Intelligence**.